# UQTS-2026: Multi-Modal Fusion Lab
## Phase 2: LSTM + ViT Late Fusion

This notebook demonstrates the finalized multi-modal architecture:
1. **AlphaUniverse**: Generates 63-day sliding windows.
2. **Sequential Stream**: LSTM processing 1D fractionally differenced returns.
3. **Spatial Stream**: Lightweight Vision Transformer (ViT) processing 2D spectrograms.
4. **Late Fusion**: Concatenating embeddings for Ranking.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datetime import datetime, timedelta

from research_lab.alpha_universe import AlphaUniverse
from research_lab.alpha_ranker import MultiModalRankNet

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = [12, 8]

### 1. Multi-Modal Dataset Generation
We use `AlphaUniverse` to extract aligned sequences and spectrograms.

In [ ]:
universe = AlphaUniverse()
tickers = ['AAPL', 'MSFT', 'GOOG', 'SPY']
universe.engine.generate_synthetic_pit_data(tickers, days=300)

# Fetch Dataset with 63-day lookback
as_of_date = datetime(2020, 1, 1) + timedelta(days=280)
dataset = universe.get_aligned_dataset(
    tickers=tickers, 
    as_of_date=as_of_date, 
    horizon=5, 
    lookback=63
)

print(f"Multi-Modal Samples: {len(dataset)}")
print(f"Sequential Stream Shape: {dataset.x_seq.shape} (N, T, 1)")
print(f"Spatial Stream Shape: {dataset.x_spatial.shape} (N, 1, Scales, T)")

### 2. Visualize Multi-Modal Window
Showing the dual representation for a single ticker.

In [ ]:
idx = 0
ticker = dataset.tickers[idx]
time = dataset.times[idx]

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

# 1D Sequence (LSTM Input)
ax1.plot(dataset.x_seq[idx].numpy(), color='tab:blue')
ax1.set_title(f"Sequential Stream (1D FD Returns): {ticker} at {time}")
ax1.set_ylabel("Stationary Return")

# 2D Spectrogram (ViT Input)
im = ax2.imshow(dataset.x_spatial[idx, 0].numpy(), aspect='auto', cmap='jet')
ax2.set_title(f"Spatial Stream (2D Wavelet Spectrogram): {ticker} at {time}")
ax2.set_ylabel("Log Scales")
ax2.set_xlabel("Lookback Steps (T=63)")

plt.tight_layout()
plt.show()

### 3. Multi-Modal RankNet Inference
Passing the dual-stream windows through the fused architecture.

In [ ]:
model = MultiModalRankNet(scales=8, lookback=63, hidden_dim=64)
model.eval()

with torch.no_grad():
    scores = model.predict_dataset(dataset)

results = pd.DataFrame({
    'ticker': dataset.tickers,
    'score': scores.squeeze().numpy(),
    'realized_alpha': dataset.y.numpy()
})

print("Final RankNet Multi-Modal Inference (Last 10 samples):")
display(results.tail(10))